# SH-wave modelling — Virieux (1984) quarter-plane example

Reproduces the expanding-wavefield surface plots from Virieux (1984), Figure 3,  
using the velocity-stress staggered-grid FD scheme on a constant homogeneous medium.

**Physical parameters (Table 1 of the paper)**

| Parameter | Value |
|-----------|-------|
| Shear velocity | 3 000 m/s |
| Grid spacing *dx* | 30 m |
| Grid size | 60 × 60 nodes |
| Source position | (615, 615) m |
| Source half-wavelength | 300 m  →  f₀ ≈ 5 Hz |

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa – required for projection='3d'

from devito import TimeFunction, NODE

from examples.seismic import demo_model, AcquisitionGeometry
from examples.seismic.sh.operators import ForwardOperator

## 1. Model

In [ ]:
shape       = (60, 60)   # grid nodes (physical domain)
spacing     = (30., 30.) # [m]
nbl         = 150         # absorbing boundary layers
space_order = 6          # 2nd-order FD as in Virieux (1984)

model = demo_model(
    'constant-sh',
    vs      = 3.0,       # shear velocity [km/s]  (= 3 000 m/s)
    shape   = shape,
    spacing = spacing,
    nbl     = nbl,
    space_order = space_order,
)
print(f'mu           = {model.mu}  (shear modulus)')
print(f'b            = {model.b}   (buoyancy = 1/density)')
print(f'critical_dt  = {model.critical_dt:.3f} ms')
print(f'domain size  = {model.domain_size} m')

## 2. Geometry and time axis

In [ ]:
t0, tn = 0., 750.  # [ms]  (0 → 0.75 s)
f0     = 0.005     # [kHz] = 5 Hz  →  half-wavelength = 3 000 / (2×5) = 300 m

# geometry.src is a property that creates a new SparseTimeFunction from
# src_positions on every access, so coordinates must be fixed at construction.
src_positions = np.array([[615., 615.]], dtype=np.float32)  # Virieux (1984), Table 1
rec_positions = np.array([np.linspace(15, 1785, num=60) for i in range(2)], dtype=np.float32).T  # observer position
geometry = AcquisitionGeometry(
    model, rec_positions, src_positions,
    t0=t0, tn=tn, src_type='Ricker', f0=f0,
)

dt = geometry.dt
print(f'Source at : {geometry.src_positions[0]} m')
print(f'dt        : {dt:.3f} ms   nt = {geometry.nt}')

## 3. Build operator and wavefield buffers

In [ ]:
op = ForwardOperator(model, geometry, space_order=space_order)

x, z = model.grid.dimensions

# Wavefield buffers (time_order=1 → 2-slot ring buffer)
v      = TimeFunction(name='v',      grid=model.grid, space_order=space_order,
                      time_order=1, staggered=NODE)
tau_xy = TimeFunction(name='tau_xy', grid=model.grid, space_order=space_order,
                      time_order=1, staggered=(x,))
tau_zy = TimeFunction(name='tau_zy', grid=model.grid, space_order=space_order,
                      time_order=1, staggered=(z,))
rec = geometry.new_rec(name='rec')

## 4. Run in segments — capture v at each snapshot time

In [ ]:
snap_ms     = [250., 500., 750.]               # snapshot times [ms]
snap_steps  = [int(t / dt) for t in snap_ms]  # corresponding time-loop indices

print('Snapshot steps:', snap_steps)

snapshots = []
prev = 0

for step in snap_steps:
    op.apply(
        v=v, tau_xy=tau_xy, tau_zy=tau_zy,
        src=geometry.src, rec=rec,
        time_m=prev, time_M=step,
        dt=dt,
        **model.physical_params(),
    )
    # With time_order=1, v.forward at iteration k is stored at slot (k+1)%2.
    # After running to time_M=step the most recent v is at slot (step+1)%2.
    slot = (step + 1) % 2
    data = v.data[slot, nbl:-nbl, nbl:-nbl].copy()
    snapshots.append(data)
    prev = step + 1
    print(f't ≈ {step*dt:.0f} ms  |v|_max = {np.abs(data).max():.4f}')

## 5. Three-panel 3-D surface plot

In [ ]:
snap_titles = ['0.25 sec', '0.50 sec', '0.75 sec']

# Physical-domain coordinate grids [m]
x_coord = np.arange(shape[0]) * spacing[0]
z_coord = np.arange(shape[1]) * spacing[1]
X, Z = np.meshgrid(x_coord, z_coord, indexing='ij')

fig = plt.figure(figsize=(18, 6), facecolor='white')

for n, (data, title) in enumerate(zip(snapshots, snap_titles)):
    ax = fig.add_subplot(1, 3, n + 1, projection='3d')

    # Normalise per-frame so the wavefront is visible at each time
    vmax = np.max(np.abs(data))
    if vmax > 0:
        data = -data / vmax

    ax.plot_surface(
        X, Z, data* 3e2,
        rstride=1, cstride=1,
        linewidth=0.3, edgecolors='k',
        color='white', shade=False,
        alpha=0.5,
    )
    ax.scatter3D(rec.coordinates.data[:, 0], rec.coordinates.data[:, 1], rec.coordinates.data[:, 0]*0)
    ax.set_title(title, fontsize=13, pad=8)
    ax.view_init(elev=30, azim=225)
    ax.set_axis_off()
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Physical-domain extent [m]
Nx = shape[0]
dx = spacing[0]
Nz_sub = shape[1]
x_min, x_max = model.origin[0], model.origin[0] + (Nx - 1) * dx          # 0 … 750 m
z_min, z_max = model.origin[1], model.origin[1] + (Nz_sub - 1) * dx  # -3.75 … 750 m

# imshow extent=(left, right, bottom, top)  — z increases downward
extent = [x_min, x_max, z_max, z_min]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)

for ax, data, t_ms in zip(axes.flat, snapshots, snap_titles):
    vmax = np.abs(data).max() or 1.0
    im = ax.imshow(
        data,
        aspect='equal',
        extent=extent,
        cmap='RdBu',
        vmin=-vmax, vmax=vmax,
        interpolation='nearest',
    )
    ax.axhline(0., color='k', linewidth=1.2, linestyle='--', label='free surface (z=0)')
    # ax.plot(src_x, src_z, 'k*', markersize=9, label='source')
    ax.set_title(f't = {t_ms} s', fontsize=12)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('z [m]')
    # plt.colorbar(im, ax=ax, shrink=0.75, label='v [m/s]')

# axes.flat[0].legend(fontsize=8, loc='lower right')
fig.suptitle('SH wavefield snapshots — IVF free surface (Pan et al. 2018)', fontsize=14)
plt.show()

In [ ]:
assert np.isclose(np.linalg.norm(rec.data), 12.0109)
print(rec.data.shape)

In [ ]:
# np.save("sh_constant_rec.npy", rec.data)
plt.imshow(rec.data)

In [ ]:
print(op)